# AlphaGenome: Non-Coding Variant Analysis

## Overview
This notebook uses DeepMind's AlphaGenome model to predict the functional impact of non-coding genetic variants. The code has been adapted from the methodology described in [Nature (2025)](https://www.nature.com/articles/s41586-025-10014-0).


AlphaGenome is a deep learning model that predicts how genetic variants affect gene regulation across different cell types. This notebook provides adapted workflows for:

1. **Single Variant Analysis**: Investigate the impact of a single variant across multiple genomic features (CAGE, DNase, RNA-Seq, ChIP-Seq, splice sites, etc.)
2. **Region-Specific Analysis**: Focus on the impact of variants in specific genomic regions, in this instance the canonical and non canonical ERG promoter
3. **Batch Analysis**: Score and visualise multiple variants from a VCF file 

## Key Features
- Scores variants across 10 different genomic features (CAGE, DNase, RNA-Seq, ChIP histone, ChIP TF, splice sites, splice junctions, splice site usage, contact maps, PRO-CAP)
- Predicts effects in a cell type specific manner using Cell Ontology terms
- Generates publication-quality plots comparing reference vs. alternate alleles
- Exports scores to CSV for further analysis

## Requirements
- **Python 3.11** 
- **AlphaGenome API key** (register at https://deepmind.google.com/science/alphagenome/)
- **Conda/Mamba** environment manager (see Setup section below)

## Setup Instructions
Run these commands in your terminal **once** to create the environment and kernel. After setup, you can select the 'alphagenome' kernel in VS Code and run subsequent notebooks without re-running setup.

```bash
# Navigate to your environments directory
cd /path/to/your/envs

# Load miniforge module
module load miniforge/3
miniforge-setup
eval "$(~/miniforge3/bin/conda shell.bash hook)"

# Configure conda
conda config --set auto_activate_base false
conda config --remove channels defaults
conda config --add channels conda-forge

# Create and activate environment
conda create -p ./alphagenome-env python=3.11
conda activate /path/to/alphagenome-env

# Install packages
pip install alphagenome
conda install anaconda::jupyter
conda install anaconda::ipykernel

# Register kernel with Jupyter
python -m ipykernel install --user --name=alphagenome --display-name \"Python (alphagenome)\"
```

After setup, select the 'Python (alphagenome)' kernel in VS Code's top-right corner before running the notebook.


In [ ]:
# In python kernel - top right, select python kernel then alphagenome-env
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 
import os

# Define folder for saving plots - create folders for each variant as saving figures will over write each time
#import os
#save_dir = "path/to/your/directory" 
#os.makedirs(save_dir, exist_ok=True)

In [ ]:
# ============================================================================
# SECTION 2: API SETUP AND MODEL INITIALIZATION
# ============================================================================
# To use AlphaGenome, obtain a free API key from:
# https://deepmind.google.com/science/alphagenome/
# Replace 'GENERATE_OWN_API_KEY' with your actual API key string.
# Initialize the DNA model client for making variant predictions.

# To use AlphaGenome, you need to generate an API key, which you can do https://deepmind.google.com/science/alphagenome/. 
# You can then use the key to access the model
API_KEY = "GENERATE_OWN_API_KEY" 
# get the model
dna_model = dna_client.create(API_KEY)

In [ ]:
# define the variant you want to investigate
variant = genome.Variant(
    chromosome='chr21',
    position=38460352,
    reference_bases='C',  # Can differ from the true reference genome base.
    alternate_bases='T',
)

# Because AlphaGenome only makes predictions on DNA sequences of specific sizes, you need to set an interval around your sequence that matches one of the sizes 
# accepted sizes: 'SEQUENCE_LENGTH_2KB', 'SEQUENCE_LENGTH_16KB', 'SEQUENCE_LENGTH_100KB', 'SEQUENCE_LENGTH_500KB', 'SEQUENCE_LENGTH_1MB'
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

In [ ]:
# This will save the predictions to a csv file where you can see cell ontology terms and select those that are relevant to your disease model. 

variant_scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=list(variant_scorers.RECOMMENDED_VARIANT_SCORERS.values()),
)

save_dir = "path/to/your/directory"
os.makedirs(save_dir, exist_ok=True)

df_scores = variant_scorers.tidy_scores(variant_scores)
csv_path = os.path.join(
    save_dir,
    f"{variant.chromosome}_{variant.position}_{variant.reference_bases}_{variant.alternate_bases}_scores.csv"
)
df_scores.to_csv(csv_path, index=False)

In [ ]:
# You can then choose the type of variant output you want and from what cell types:
variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[
        dna_client.OutputType.CAGE,
        dna_client.OutputType.DNASE,
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.CHIP_HISTONE,
        dna_client.OutputType.CHIP_TF,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
        dna_client.OutputType.SPLICE_SITES, 
        dna_client.OutputType.CONTACT_MAPS,
        dna_client.OutputType.PROCAP,
    ],
    ontology_terms=['CL:2000010', 'CL:2000011', 'CL:2000041', 'CL:0000115', 'CL:1000413', 'CL:2000018', 'CL:0002138', 'CL:0002618', 'CL:0002543' , 'CL:0002554'],
)

#Ontology term corresponds to endothelial and lymphatic cells. 
#CL:2000010	dermis blood vessel endothelial cell
#CL:2000011	dermis lymphatic vessel endothelial cell
#CL:2000041	dermis microvascular lymphatic vessel endothelial cell
#CL:0000115	endothelial cell
#CL:1000413	endothelial cell of artery
#CL:2000018	endothelial cell of coronary artery
#CL:0002138	endothelial cell of lymphatic vessel
#CL:0002618	endothelial cell of umbilical vein
#CL:0002543	vein endothelial cell
#CL:0002554	fibroblast of lymphatic vessel

In [ ]:
#Before plotting the results, transcript extractor needs to be defined:
#The GTF file contains information on the location of all transcripts.
#Note that we use genome assembly hg38 for human.
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Set up transcript extractors using the information in the GTF file.
# Keep only protein-coding transcripts
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)

# Filter to canonical transcripts (basic/CCDS) - we want Canonical of ERG
gtf_transcripts = gtf_transcripts[
    gtf_transcripts['tag'].str.contains("basic|CCDS", na=False)
]
# If you want to filter to for longer ERG trasncripts when variants are in the longer isoform of ERG
# gtf_transcripts = gene_annotation.filter_to_longest_transcript(gtf_transcripts)

# Create the transcript extractor
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

In [ ]:
#Extract canonical transcripts for plotting
canonical_transcripts = transcript_extractor.extract(interval)

#Extract longest trasncript for plotting
#longest_transcripts = transcript_extractor.extract(interval)

In [ ]:
#Plot figures

# Helper function for safe filenames
def safe_filename(name):
    return name.lower().replace(" ", "_") + ".png"

# Define a plotting function 
def plot_and_save(track_name, ref_track, alt_track):
    if ref_track is None or alt_track is None:
        print(f"Skipping {track_name}: data not available")
        return

    plot_components.plot(
        [
            plot_components.TranscriptAnnotation(canonical_transcripts), #change to longest_transcripts if need predicitons based on longer ERG isoform
            plot_components.OverlaidTracks(
                tdata={"REF": ref_track, "ALT": alt_track},
                colors={"REF": "#000000", "ALT": "#E69F00"},
            ),
        ],
        interval=variant_output.reference.rna_seq.interval.resize(2**15),
        annotations=[plot_components.VariantAnnotation([variant], alpha=0.8)],
    )

    plt.title(f"{track_name} — REF vs ALT")
    plt.savefig(os.path.join(save_dir, safe_filename(track_name)), dpi=300, bbox_inches="tight")
    plt.show()


# Plot all 1D tracks
plot_and_save("CAGE", variant_output.reference.cage, variant_output.alternate.cage)
plot_and_save("DNase", variant_output.reference.dnase, variant_output.alternate.dnase)
plot_and_save("RNA-Seq", variant_output.reference.rna_seq, variant_output.alternate.rna_seq)
plot_and_save("ChIP-Seq Histone", variant_output.reference.chip_histone, variant_output.alternate.chip_histone)
plot_and_save("ChIP-Seq TF", variant_output.reference.chip_tf, variant_output.alternate.chip_tf)
plot_and_save("Splice Sites", variant_output.reference.splice_sites, variant_output.alternate.splice_sites)
plot_and_save("Splice Site Usage", variant_output.reference.splice_site_usage, variant_output.alternate.splice_site_usage)
plot_and_save("PRO-CAP", variant_output.reference.procap, variant_output.alternate.procap)

## ERG Promoter Region Analysis

This section analyzes variant effects specifically within the ERG promoter region. Rather than examining the entire 1MB interval, we zoom in on the promoter to visualize precise effects on transcription initiation (CAGE) and PRO-CAP signals.

**Note**: AlphaGenome can only generate predictions within 1MB of the variant. Ensure your variant is within this distance of the promoter of interest.


In [ ]:
# This section uses the alphagenome model however visualises the variant impact around the promoter region only, for the CAGE and PROCAP tracks.
# If the variant is further than 500kb from the promoter region the model will not be able to generate a prediction  

# DEFINE THE VARIANT
variant = genome.Variant(
    chromosome='chr21',
    position=38381934,
    reference_bases='T',  
    alternate_bases='TA',
)

interval = variant.reference_interval.resize(
    dna_client.SEQUENCE_LENGTH_1MB
)
# SCORE THE VARIANT
variant_scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=list(
        variant_scorers.RECOMMENDED_VARIANT_SCORERS.values()
    ),
)

save_dir = "path/to/your/directory"
os.makedirs(save_dir, exist_ok=True)

df_scores = variant_scorers.tidy_scores(variant_scores)

csv_path = os.path.join(
    save_dir,
    f"{variant.chromosome}_{variant.position}_{variant.reference_bases}_{variant.alternate_bases}_scores.csv",
)

df_scores.to_csv(csv_path, index=False)


# GENERATE PREDICTIONS

variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[
        dna_client.OutputType.CAGE,
        dna_client.OutputType.PROCAP,
    ],
    ontology_terms=[
        "CL:2000010", "CL:2000011", "CL:2000041", "CL:0000115",
        "CL:1000413", "CL:2000018", "CL:0002138",
        "CL:0002618", "CL:0002543", "CL:0002554"
    ],
)

#Ontology term corresponds to endothelial and lymphatic cells. 
#CL:2000010	dermis blood vessel endothelial cell
#CL:2000011	dermis lymphatic vessel endothelial cell
#CL:2000041	dermis microvascular lymphatic vessel endothelial cell
#CL:0000115	endothelial cell
#CL:1000413	endothelial cell of artery
#CL:2000018	endothelial cell of coronary artery
#CL:0002138	endothelial cell of lymphatic vessel
#CL:0002618	endothelial cell of umbilical vein
#CL:0002543	vein endothelial cell
#CL:0002554	fibroblast of lymphatic vessel

# DEFINE ERG PROMOTER
# remove hashtags depending on which ERG promoter you are investigating  

# Canonical ERG promoter
erg_promoter_interval = genome.Interval(
    chromosome="chr21",
    start=38496138,
    end=38500210,
)

# non canonical ERG promoter
#erg_promoter_interval = genome.Interval(
 #   chromosome="chr21",
  #  start=38660260,
   # end=38661831,
#)



# LOAD GTF AND EXTRACT TRANSCRIPTS

gtf = pd.read_feather(
    "https://storage.googleapis.com/alphagenome/reference/gencode/"
    "hg38/gencode.v46.annotation.gtf.gz.feather"
)

gtf_transcripts = gene_annotation.filter_protein_coding(gtf)

gtf_transcripts = gtf_transcripts[
    gtf_transcripts["tag"].str.contains("basic|CCDS", na=False)
]

transcript_extractor = transcript_utils.TranscriptExtractor(
    gtf_transcripts
)

canonical_transcripts_promoter = transcript_extractor.extract(
    erg_promoter_interval
)

# PLOT

def safe_filename(name):
    return name.lower().replace(" ", "_").replace("—", "_") + ".png"


def plot_and_save(track_name, ref_track, alt_track):

    if ref_track is None or alt_track is None:
        print(f"Skipping {track_name}: data not available")
        return

    plot_components.plot(
        [
            plot_components.TranscriptAnnotation(
                canonical_transcripts_promoter
            ),
            plot_components.OverlaidTracks(
                tdata={"REF": ref_track, "ALT": alt_track},
                colors={
                    "REF": "000000",
                    "ALT": "#D55E00",
                },
            ),
        ],
        interval=erg_promoter_interval,
        annotations=[],
    )

    plt.title(f"{track_name} — ERG Promoter (REF vs ALT)")

    out_path = os.path.join(
        save_dir, safe_filename(track_name)
    )

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved: {out_path}")


# PLOT ONLY PROMOTER TRACKS

plot_and_save(
    "ERG Promoter — CAGE-38499489",
    variant_output.reference.cage,
    variant_output.alternate.cage,
)

plot_and_save(
    "ERG Promoter - PRO-CAP-38499489",
    variant_output.reference.procap,
    variant_output.alternate.procap,
)


## Batch Variant Analysis

Process multiple variants from a VCF file in a single run. This workflow scores all variants and generates plots for each one.

**Important Limitations**:
- Script can only process 30 variants before timing out
- For larger datasets, split your VCF file into batches of ≤30 variants and re-run this section multiple times
- The VCF file should have variants numbered (ID column) - these IDs are used to name output files


In [ ]:
# ============================================================================
# SECTION 2: API SETUP AND MODEL INITIALISATION
# ============================================================================
# To use AlphaGenome, obtain a free API key from:
# https://deepmind.google.com/science/alphagenome/
# Replace 'GENERATE_OWN_API_KEY' with your actual API key string.
# Initialize the DNA model client for making variant predictions.

# This section uses the alphagenome model however has been adapted to score all variants at once. THE VCF file needs to have the variants numbered and this will correspond to the file name for each variant prediction
# The model can only score 30 variants before timing out. Make VCF files containing 30 variants and re-run code however many times necessary.

#Setup alphagenome
import os
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from alphagenome import colab_utils
from alphagenome.data import genome, gene_annotation, transcript as transcript_utils
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components

API_KEY = "GENERATE_OWN_API_KEY" 
dna_model = dna_client.create(API_KEY)

#Input VCF file
vcf_file = "path/to/your/file" 

#Create folder to save results
save_dir = "path/to/your/directory"
os.makedirs(save_dir, exist_ok=True)

#Specify sequence length and organism 
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS['SEQUENCE_LENGTH_1MB']
organism = dna_client.Organism.HOMO_SAPIENS

#Use all recommended AlphaGenome scores 
all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
selected_scorers = list(all_scorers.values())

#Target ontologies to keep in final CSV
ontology_terms = {
    'CL:2000010','CL:2000011','CL:2000041','CL:0000115','CL:1000413',
    'CL:2000018','CL:0002138','CL:0002618','CL:0002543','CL:0002554'
}


#Load GTF and extract canonical transcript
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/hg38/gencode.v46.annotation.gtf.gz.feather'
)
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gtf_transcripts[gtf_transcripts['tag'].str.contains("basic|CCDS", na=False)]
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)


#Read VCF
vcf = pd.read_csv(vcf_file, comment="#", sep="\t", header=None,
                  names=["CHROM","POS","ID","REF","ALT","QUAL","FILTER","INFO"])
vcf['variant_id'] = vcf['ID'].astype(str)  # ensure string type for folder names

#Batch scoring of variants
all_results = []

for _, row in tqdm(vcf.iterrows(), total=len(vcf), desc="Scoring variants"):
    variant = genome.Variant(
        chromosome=str(row.CHROM),
        position=int(row.POS),
        reference_bases=row.REF,
        alternate_bases=row.ALT,
        name=str(row.variant_id)
    )
    interval = variant.reference_interval.resize(sequence_length)
    
    # Score variant
    variant_scores_list = dna_model.score_variant(
        interval=interval,
        variant=variant,
        variant_scorers=selected_scorers,
        organism=organism
    )
    all_results.append(variant_scores_list)


    # Extract canonical transcripts for csv
    canonical_transcripts = transcript_extractor.extract(interval)
    
#Save scores related to cells of interest for disease model 
df_scores = variant_scorers.tidy_scores(all_results)
df_scores_filtered = df_scores[df_scores['ontology_curie'].isin(ontology_terms)]
csv_path = os.path.join(save_dir, "batch_variant_scores.filtered.csv")
df_scores_filtered.to_csv(csv_path, index=False)

In [ ]:
# LOAD GTF AND EXTRACT TRANSCRIPTS
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/hg38/gencode.v46.annotation.gtf.gz.feather'
)
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gtf_transcripts[gtf_transcripts['tag'].str.contains("basic|CCDS", na=False)]
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

# Helper functions for saving files
def safe_filename(name):
    return name.lower().replace(" ", "_") + ".png"

def plot_and_save(track_name, ref_track, alt_track, transcripts, variant_name):
    if ref_track is None or alt_track is None:
        print(f"Skipping {track_name}: data not available")
        return

    plot_components.plot(
        [
            plot_components.TranscriptAnnotation(transcripts),
            plot_components.OverlaidTracks(
                tdata={"REF": ref_track, "ALT": alt_track},
                colors={"REF": "dimgrey", "ALT": "red"},
            ),
        ],
        interval=ref_track.interval.resize(2**15),
        annotations=[plot_components.VariantAnnotation([variant], alpha=0.8)],
    )
    plt.title(f"{track_name} — REF vs ALT")
    plt.savefig(os.path.join(save_dir, f"{variant_name}_{safe_filename(track_name)}"), dpi=300, bbox_inches="tight")
    plt.close()

# PLOT
vcf = pd.read_csv(vcf_file, comment="#", sep="\t", header=None,
                  names=["CHROM","POS","ID","REF","ALT","QUAL","FILTER","INFO"])
vcf['variant_id'] = vcf['ID']

#Generate plots
track_names_and_attrs = [
    ("CAGE", "cage"),
    ("DNASE", "dnase"),
    ("RNA_SEQ", "rna_seq"),
    ("CHIP_HISTONE", "chip_histone"),
    ("CHIP_TF", "chip_tf"),
    ("SPLICE_SITES", "splice_sites"),
    ("SPLICE_SITE_USAGE", "splice_site_usage"),
    ("PROCAP", "procap"),
]

for _, row in tqdm(vcf.iterrows(), total=len(vcf), desc="Plotting variants"):
    variant = genome.Variant(
        chromosome=str(row.CHROM),
        position=int(row.POS),
        reference_bases=row.REF,
        alternate_bases=row.ALT,
        name=str(row.variant_id)
    )
    interval = variant.reference_interval.resize(sequence_length)

    # Predict variant
    variant_output = dna_model.predict_variant(
        interval=interval,
        variant=variant,
        requested_outputs=[getattr(dna_client.OutputType, attr.upper()) for _, attr in track_names_and_attrs],
        ontology_terms=ontology_terms
    )

    # Extract canonical transcripts
    canonical_transcripts = transcript_extractor.extract(interval)

    # Plot each track
    for track_name, attr in track_names_and_attrs:
        ref = getattr(variant_output.reference, attr, None)
        alt = getattr(variant_output.alternate, attr, None)
        plot_and_save(track_name, ref, alt, canonical_transcripts, variant.name)

